## Item Dependency Graph Web Tool


##### This notebook creates a dependency graph json file for an item in ArcGIS Online, including upstream and downstream dependencies both inside and outside the organisation. The json file is output to the notebook files and then published as an item into ArcGIS Online. The json file can be downloaded from the item page and uploaded into the item graph visualisation tool. This notebook can be run in the notebook editor directly, or published and run as a web tool.

#### Import modules

In [ ]:
import os
import json
from arcgis.gis import GIS, Item
from arcgis.apps.itemgraph import create_dependency_graph

#### Connect to ArcGIS Online organisation

In [ ]:
gis = GIS("home")
print(f"Connected as: {gis.users.me.username}")

#### Input parameter

In [ ]:
target_item_id = ""

In [ ]:
if not target_item_id:
    raise ValueError(
        "target_item_id is empty. Provide an ArcGIS Online item ID."
    )

#### Fetch the target item

In [ ]:
target_item = Item(gis=gis, itemid=target_item_id)
print(f"Target item: {target_item.title} ({target_item.type})")

#### Output configuration

In [ ]:
output_directory = "/arcgis/home/output"
output_filename = f"{target_item_id}_dependency_graph.json"
output_file_path = os.path.join(output_directory, output_filename)

#### Item dependency graph function

In [ ]:
def convert_itemgraph_to_json(source_graph: "arcgis.apps.itemgraph.ItemGraph", excluded_item_types: set = None) -> dict:
    excluded_item_types = excluded_item_types if excluded_item_types is not None else set()
    included_item_ids = set()
    graph_nodes = []

    # Process items into nodes
    for graph_item in source_graph.all_items(out_format="item"):
        if isinstance(graph_item, Item) and graph_item.type in excluded_item_types:
            continue

        if isinstance(graph_item, Item):
            node_title_html = f"<b>{graph_item.title}</b><br/>{graph_item.type}<br/>{graph_item.id}<br/>"
            if graph_item.url and graph_item.url.strip():
                node_title_html = (
                    f"<b><a href='{graph_item.url}' target='_blank'>{graph_item.title}</a></b>"
                    f"<br/>{graph_item.type}<br/>{graph_item.id}<br/>"
                )

            graph_nodes.append(
                {
                    "id": str(graph_item.id),
                    "name": f"{graph_item.title} ({graph_item.type})",
                    "title": node_title_html,
                    "type": graph_item.type,
                    "group": graph_item.type,
                }
            )
            included_item_ids.add(graph_item.id)
        else:
            # graph_item is assumed to be a string, typically a URL representing an external/unresolved dependency
            node_title_html = f"<b><a href='{graph_item}' target='_blank'>{graph_item}</a></b>"
            graph_nodes.append(
                {
                    "id": str(graph_item),
                    "name": str(graph_item),
                    "title": node_title_html,
                    "type": "Unknown",
                    "group": "Unknown",
                }
            )
            included_item_ids.add(graph_item)

    # Process edges into links
    graph_links = []
    for source_id, destination_id in source_graph.edges:
        if (
            source_id in included_item_ids
            and destination_id in included_item_ids
            and str(source_id) != str(destination_id)
        ):
            graph_links.append({"source": str(source_id), "target": str(destination_id)})

    return {"nodes": graph_nodes, "links": graph_links}

#### Build the item dependency graph

In [ ]:
print("Building item dependency graph...")
dependency_graph = create_dependency_graph(
    gis=gis,
    item_list=[target_item],
    outside_org=True,
    include_reverse=True,
)

#### Convert to D3.js json

In [ ]:
dependency_graph_json = convert_itemgraph_to_json(source_graph=dependency_graph)

print(f"Nodes: {len(dependency_graph_json['nodes'])}")
print(f"Links: {len(dependency_graph_json['links'])}")

#### Write the JSON

In [ ]:
os.makedirs(output_directory, exist_ok=True)
with open(output_file_path, "w") as output_file:
    json.dump(dependency_graph_json, output_file)

print(f"Dependency graph written to: {output_file_path}")

#### Publish json file as item in ArcGIS Online

In [ ]:
json_item_properties = {
    "title": f"Dependency Graph JSON - {target_item_id}",
    "type": "CSV", # Accepted item type for json files
    "tags": "item dependency, item graph, json",
}

root_folder = gis.content.folders.get()

uploaded_json_item = root_folder.add(
    item_properties=json_item_properties,
    file=output_file_path,
).result()

json_item_page_url = uploaded_json_item.homepage

print(f"Published as Content item: {uploaded_json_item.id}")
print(f"Item page: {uploaded_json_item.homepage}")

#### Output parameter for web tool

In [ ]:
# Only run this snippet during web tool execution when env variable "ENB_JOBID" will be present

if "ENB_JOBID" in os.environ:
    out_param_name = "json_item_page_url"
    out_param_file = os.path.join(
        os.environ["ENB_JOBID"], "value_" + out_param_name + ".dat")
    with open(out_param_file, "w") as writer:
        writer.write(str(json_item_page_url))